## Visualization 3

In [29]:
## Scientific calculus
import pandas as pd
import numpy as np

## Plotting libraries
import plotly.graph_objects as go

### Import Dataset and recover POKE_dfs

In [30]:
POKE_df = pd.read_csv("../DATASETS/pokemon_complete_2025.csv")

roman_to_int = {
    'I': 1, 'II': 2, 'III': 3, 'IV': 4, 'V': 5,
    'VI': 6, 'VII': 7, 'VIII': 8, 'IX': 9
}

# Map actual publication years
gen_to_year = {
    'I': 1996, 'II': 1999, 'III': 2002, 'IV': 2006, 'V': 2010,
    'VI': 2013, 'VII': 2016, 'VIII': 2019, 'IX': 2022
}

POKE_df['generation_num'] = POKE_df['generation'].map(roman_to_int)
POKE_df = POKE_df.dropna(subset=['generation_num'])

metrics = [
    'base_stat_total', 'attack', 'defense', 'hp', 'speed', 
    'hatch_counter', 'is_dual_type', 'weight_kg', 'height_m', "capture_rate"
]

# Choose the gens to compare for the radar charts
genA = "I"
genB = "IX"
year_genA = gen_to_year[genA]
year_genB = gen_to_year[genB]

## FONTS and color choices
poke_font = "Pokemon Emerald, Arial"
fontsizetext=20
fontsizetitle =32

# Anchor colors for the radial plot and highlight boxes
color_genA = '#e33c12'  # Red
color_genB = '#4e49de'  # Blue-purple

# Distinct palette for the line chart stats
attack_color  = "#F57C00" # Bold Orange
defense_color = "#00897B" # Deep Teal
hp_color      = "#7CB342" # Vibrant Green
speed_color   = "#D81B60" # Magenta/Pink

# Group by generation, KEEPING generation_num to sort chronologically for the line chart
gen_means = POKE_df.groupby(['generation', 'generation_num'], observed=False)[metrics].mean().reset_index()
gen_means = gen_means.sort_values('generation_num') 

# Assign the publication year to our grouped dataframe
gen_means['publication_year'] = gen_means['generation'].map(gen_to_year)

max_values = gen_means[metrics].max().values
genA_data = gen_means[gen_means['generation'] == genA][metrics].iloc[0].values
genB_data = gen_means[gen_means['generation'] == genB][metrics].iloc[0].values
genA_norm = genA_data / max_values
genB_norm = genB_data / max_values

display_metrics = [
    "Base Stats", 
    f"<span style='color:{attack_color}'>Attack</span>", 
    f"<span style='color:{defense_color}'>Defense</span>", 
    f"<span style='color:{hp_color}'>HPs</span>", 
    f"<span style='color:{speed_color}'>Speed</span>", 
    "Hatch Counter", 
    "Dual type", 
    "Weight", 
    "Height", 
    "Capture Rate"
]
display_metrics_closed = display_metrics + [display_metrics[0]]

genA_norm_closed = np.append(genA_norm, genA_norm[0])
genB_norm_closed = np.append(genB_norm, genB_norm[0])

genA_text = [f"{v:.1f}" for v in np.append(genA_data, genA_data[0])]
genB_text = [f"{v:.1f}" for v in np.append(genB_data, genB_data[0])]

# Text positioning logic
out_map = [
    'top center', 'top right', 'middle right', 'bottom right', 
    'bottom center', 'bottom center', 'bottom left', 'middle left', 
    'top left', 'top left'
]
out_map.append(out_map[0]) 

opp_map = [
    'bottom center', 'bottom left', 'middle left', 'top left', 
    'top center', 'top center', 'top right', 'middle right', 
    'bottom right', 'bottom right'
]
opp_map.append(opp_map[0]) 

pos_genA = []
pos_genB = []

for i, (gA, gB) in enumerate(zip(genA_norm_closed, genB_norm_closed)):
    ranks = [(gA, pos_genA), (gB, pos_genB)]
    ranks.sort(key=lambda x: x[0], reverse=True)
    
    ranks[0][1].append(out_map[i])         
    ranks[1][1].append(opp_map[i])   

### Building Figures

In [33]:
fig = go.Figure()

## ........................... ##
## LINE CHART metrics and plot ##
## ........................... ##
line_metrics = {
    'attack': {'label': 'Attack', 'color': attack_color},
    'defense': {'label': 'Defense', 'color': defense_color},
    'hp': {'label': 'HP', 'color': hp_color},
    'speed': {'label': 'Speed', 'color': speed_color}
}

# Add a scatterplot spaced by year
for metric_key, config in line_metrics.items():
    raw_trend = gen_means[metric_key]
    
    fig.add_trace(
        go.Scatter(
            x=gen_means['publication_year'], 
            y=raw_trend,
            mode='lines+markers',
            name=config['label'],
            line=dict(color=config['color'], width=2),
            marker=dict(size=6, symbol='circle', color=config['color']),
            fillcolor="#FFFFFF",
            legend="legend2"  
        )
    )

## Define the layout, x label: be able also to insert the year
generations_order = ['I', 'II', 'III', 'IV', 'V', 'VI', 'VII', 'VIII', 'IX']
x_tickvals = [gen_to_year[g] for g in generations_order]
x_ticktext = []
for g in generations_order:
    yr = gen_to_year[g]
    if g == genA:
        x_ticktext.append(f"<b style='color:{color_genA}'>GEN {g}<br>({yr})</b>")
    elif g == genB:
        x_ticktext.append(f"<b style='color:{color_genB}'>GEN {g}<br>({yr})</b>")
    else:
        x_ticktext.append(f"{g}<br>({yr})")


## ................... ##
## Add the Radar Plot ##
## ................... ##

## Add the radar values for the first compared generation
fig.add_trace(
    go.Scatterpolar(
        r=genA_norm_closed,
        theta=display_metrics_closed,
        fill='toself',
        fillcolor='rgba(227, 60, 18, 0.15)',
        line=dict(color=color_genA, width=3),
        mode='lines+markers+text',
        text=genA_text,
        textposition=pos_genA,
        textfont=dict(family=poke_font, size=14, color=color_genA),
        name=f'Gen {genA}',
        showlegend=False,
        legend="legend" 
    )
)

## Add the radar values for the second compared generation
fig.add_trace(
    go.Scatterpolar(
        r=genB_norm_closed,
        theta=display_metrics_closed,
        fill='toself',
        fillcolor='rgba(78, 73, 222, 0.15)', 
        line=dict(color=color_genB, width=3),
        mode='lines+markers+text',
        text=genB_text,
        textposition=pos_genB,
        textfont=dict(family=poke_font, size=14, color=color_genB),
        name=f'Gen {genB}',
        showlegend=False,
        legend="legend" 
    )
)


## ................................ ##
## Figure Update  and general infos ##
## ................................ ##
fig.update_layout(

    ## TITLE and general layout 
    title=dict(
        text="<b>AN ALMOST THIRTY YEARS LONG PARAMETER TUNING</b><br><sup>Evolution of core base statistics alongside specific generation snapshots</sup>",
        font=dict(family=poke_font, size=32, color="#5A5665"),
        pad=dict(b=0), 
        x=0.065,
        xanchor="left",
        y=0.95,
        yanchor="top"
    ),
    paper_bgcolor="rgba(255, 255, 255, 0)", 
    plot_bgcolor="#FFFFFF", 
    margin=dict(l=80, r=80, t=150, b=220), 
    width=1200, 
    height=950, 

    ## RADAR PLOT
    polar=dict(
        bgcolor="#FFFFFF", 
        domain=dict(x=[0.20, 0.80], y=[0.39, 0.95]), 
        radialaxis=dict(
            visible=True, range=[0, 1.25], showticklabels=False, 
            showline=False, gridcolor="#D9DCE0", gridwidth=2, griddash="dot"
        ),
        angularaxis=dict(
            tickfont=dict(family=poke_font, size=19, color="#5A5665"), 
            linecolor="#6D6C71", linewidth=1, gridcolor="#D9DCE0",
            gridwidth=2, griddash="dot", direction="clockwise"
        )
    ),
    
    ## Legend
    legend=dict(
        font=dict(family=poke_font, size=18, color="#5A5665"),
        yanchor="middle", y=0.72, 
        xanchor="left", x=0.05, 
        bgcolor="#FFFFFF", bordercolor="#5A5665", borderwidth=0,
        title_text="Generations",
        title_font=dict(family=poke_font, size=18)
    ),
    
    legend2=dict(
        font=dict(family=poke_font, size=16, color="#5A5665"),
        orientation="h", 
        yanchor="bottom", y=0.02,  
        xanchor="center", x=0.5,
        bgcolor="#FFFFFF",         
        bordercolor="#D9DCE0",     
        borderwidth=1
    ),

    ## XAXIS
    xaxis=dict(
        title=dict(
            text="Generation (Scaled by Publication Year)",
            font=dict(family=poke_font, size=20, color="#5A5665")
        ),
        range=[1994.6, 2023.4], 
        tickmode='array',
        tickvals=x_tickvals,
        ticktext=x_ticktext, # Injected dynamic text here
        tickfont=dict(family=poke_font, size=16, color="#5A5665"),
        showgrid=False, gridcolor="rgba(176, 184, 192, 0.1)", gridwidth=1,
        ticks="outside", ticklen=10, #showline=True, linecolor="#6d687d", 
        linewidth=3, mirror=True, zeroline=False
    ),
    ## YAXIS
    yaxis=dict(
        title=dict(
            text="Average Stats Values", 
            font=dict(family=poke_font, size=20, color="#5A5665")
        ),
        domain=[0.0, 0.30],  
        range=[45, 95],  
        tickfont=dict(family=poke_font, size=16, color="#5A5665"),
        showgrid=False, gridcolor="rgba(176, 184, 192, 0.1)", gridwidth=2, griddash="dot",
        ticks="outside", ticklen=10, dtick=10, 
        #showline=True, linecolor="#6d687d", 
        linewidth=3, mirror=True, zeroline=False
    )
)


## ................................................. ##
## CUSTOM GRID to see first and last gen in the plot ##
## ................................................. ##
custom_x_interval = [1994, 2026] 
grid_y_positions = [60, 70, 80, 90] 

for y_val in grid_y_positions:
    fig.add_shape(
        type="line",
        x0=custom_x_interval[0], x1=custom_x_interval[1],
        y0=y_val, y1=y_val,
        line=dict(
            color="rgba(176, 184, 192, 0.3)",
            width=1,
            dash="dot"
        ),
        layer="below"
    )

# HIGHLIGHT BOXES strictly mapped to years
fig.add_shape(
    type="rect",
    x0=year_genA - 1.2, x1=year_genA + 1.2,
    y0=47, y1=93, 
    fillcolor='rgba(227, 60, 18, 0.15)', 
    line=dict(color=color_genA, width=2), 
    layer="below"
)

fig.add_shape(
    type="rect",
    x0=year_genB - 1.2, x1=year_genB + 1.2,
    y0=47, y1=93, 
    fillcolor='rgba(78, 73, 222, 0.15)', 
    line=dict(color=color_genB, width=2), 
    layer="below"
)

## ........................ ##
## CAPTION Box and its text ##
## ........................ ##
def rounded_rect(x0, y0, x1, y1, rx, ry):
    return (f"M {x0+rx},{y0} L {x1-rx},{y0} Q {x1},{y0} {x1},{y0+ry} "
            f"L {x1},{y1-ry} Q {x1},{y1} {x1-rx},{y1} L {x0+rx},{y1} "
            f"Q {x0},{y1} {x0},{y1-ry} L {x0},{y0+ry} Q {x0},{y0} {x0+rx},{y0} Z")

cap_x0, cap_x1 = 0.0, 1.0  
cap_y0, cap_y1 = -0.35, -0.20
dx_black, dy_black = 0.002, 0.004
dx_red, dy_red = 0.015, 0.008
rx_out, ry_out = 0.015, 0.035

fig.add_shape(type="path", xref="paper", yref="paper", path=rounded_rect(cap_x0, cap_y0, cap_x1, cap_y1, rx_out, ry_out), fillcolor="#000000", line_width=0, layer="below")
fig.add_shape(type="path", xref="paper", yref="paper", path=rounded_rect(cap_x0 + dx_black, cap_y0 + dy_black, cap_x1 - dx_black, cap_y1 - dy_black, rx_out * 0.9, ry_out * 0.9), fillcolor="#D64848", line_width=0, layer="below")
fig.add_shape(type="path", xref="paper", yref="paper", path=rounded_rect(cap_x0 + dx_red, cap_y0 + dy_red, cap_x1 - dx_red, cap_y1 - dy_red, rx_out * 0.6, ry_out * 0.6), fillcolor="#FFFFFF", line_width=0, layer="below")

fig.add_annotation(
    xref="paper", yref="paper", x=0.025, y=-0.26, 
    text="<b>FIGURE</b>. Line chart tracks the true average of HP, Attack, Defense, and Speed across all 9 generations. Above the chart, comprehensive radar profiles ",
    showarrow=False, xanchor="left", yanchor="bottom", align="left",   
    font=dict(family=poke_font, size=19, color="#5A5665")
)
fig.add_annotation(
    xref="paper", yref="paper", x=0.065, y=-0.257, 
    text=f"   compare Gen {genA} and {genB} averaged statistics. The highlighted boxes emphasize the specific generations showcased in the radar charts.",
    showarrow=False, xanchor="left", yanchor="top", align="left",                     
    font=dict(family=poke_font, size=19, color="#5A5665")
)

fig.add_annotation(
    xref="paper", yref="paper", x=0.065, y=-0.295, 
    text=f"   Source -  https://www.kaggle.com/datasets/darkmatternet/ultimate-pokmon-dataset-2025.",
    showarrow=False, xanchor="left", yanchor="top", align="left",                     
    font=dict(family=poke_font, size=19, color="#5A5665")
)

## .............................................. ##
## GLOBAL BORDER around both line and radar plots ##
## .............................................. ##
fig.add_shape(
    type="rect",
    xref="paper", 
    yref="paper",
    x0=-0.004,  # Slightly outside the left edge for padding
    x1=1.004,   # Slightly outside the right edge
    y0=-0.005,  # Slightly below the line plot
    y1=1.05,   # Slightly above the radar plot
    line=dict(
        color="#5A5665", # Matches your theme's text color
        width=3
    ),
    fillcolor="rgba(0,0,0,0)", # Transparent inside
    layer="above"
)

fig.add_shape(
    type="rect",
    xref="paper", 
    yref="paper",
    x0=-0.004,  # Slightly outside the left edge for padding
    x1=1.004,   # Slightly outside the right edge
    y0=-0.005,  # Slightly below the line plot
    y1=1.05,   # Slightly above the radar plot
    line=dict(
        color="#5A5665", # Matches your theme's text color
        width=3
    ),
    fillcolor="#FFFFFF", # Transparent inside
    layer="below"
)


## Download config
config_download = {
  'toImageButtonOptions': {
    'format': 'png', 
    'filename': 'Final alternative 3 highres-2',
    'height': 950,
    'width': 1200,
    'scale': 4          
  }
}
fig.show(config=config_download)